In [5]:
import numpy as np

# --- MDP Definition ---
states = ['S1', 'S2', 'S3', 'S4', 'G']
actions = {
    'S1': ['a1', 'a2'],
    'S2': ['a1', 'a2'],
    'S3': ['a1', 'a2'],
    'S4': ['a1'],
    'G': []
}

# Transition model T[s][a] = list of (next_state, probability)
T = {
    'S1': {
        'a1': [('S2', 0.8), ('S3', 0.2)],
        'a2': [('S3', 0.7), ('S4', 0.3)]
    },
    'S2': {
        'a1': [('S1', 0.5), ('S3', 0.4), ('G', 0.1)],
        'a2': [('S3', 0.9), ('S4', 0.1)]
    },
    'S3': {
        'a1': [('S2', 0.6), ('G', 0.4)],
        'a2': [('S4', 1.0)]
    },
    'S4': {
        'a1': [('G', 1.0)]
    },
    'G': {}
}

# --- Parameters ---
cost = 2
gamma = 0.9
epsilon = 1e-6

# --- Helper functions ---
def policy_evaluation(policy, V):
    """Iteratively evaluate a given policy."""
    while True:
        delta = 0
        for s in states:
            if s == 'G':
                continue
            v = V[s]
            a = policy[s]
            new_v = cost
            for s_next, p in T[s][a]:
                new_v += gamma * p * V[s_next]
            V[s] = new_v
            delta = max(delta, abs(v - new_v))
        if delta < epsilon:
            break
    return V


def policy_improvement(V):
    """Improve the policy greedily based on current value function."""
    policy_stable = True
    new_policy = {}
    for s in states:
        if s == 'G':
            new_policy[s] = None
            continue
        Q_values = {}
        for a in actions[s]:
            Q = cost
            for s_next, p in T[s][a]:
                Q += gamma * p * V[s_next]
            Q_values[a] = Q
        best_action = min(Q_values, key=Q_values.get)
        new_policy[s] = best_action
        if policy[s] != best_action:
            policy_stable = False
    return new_policy, policy_stable


# --- Policy Iteration ---
# Initialize arbitrary policy
policy = {s: ('a1' if actions[s] else None) for s in states}
V = {s: 0 for s in states}
iteration = 0

while True:
    iteration += 1
    V = policy_evaluation(policy, V)
    new_policy, stable = policy_improvement(V)
    if stable:
        break
    policy = new_policy

# --- Results ---
print("Optimal Policy (π*):")
for s in states:
    print(f"  {s}: {policy[s]}")

print("\nOptimal Value Function (V*):")
for s in states:
    print(f"  {s}: {V[s]:.4f}")

print(f"\nConverged after {iteration} policy iterations.")


Optimal Policy (π*):
  S1: a2
  S2: a2
  S3: a2
  S4: a1
  G: None

Optimal Value Function (V*):
  S1: 4.9340
  S2: 5.2580
  S3: 3.8000
  S4: 2.0000
  G: 0.0000

Converged after 2 policy iterations.
